# Chạy benchmark trên Colab

Notebook này chỉ là **trình chạy**: code nằm trong repo (`src/rsf`), kết quả ghi vào Google Drive để không mất khi hết phiên.

1. Runtime → Change runtime type → CPU là đủ (GPU chỉ giúp RNN/LSTM nhanh hơn một chút).
2. Chuẩn bị 2 file `daily_series.csv` và `promotions.csv` (sinh bằng `python -m rsf.data` trên laptop) trong thư mục Drive `MyDrive/rsf/data/`.
3. Chạy từng cell. Ngắt giữa chừng thì chạy lại: task đã xong sẽ được bỏ qua.

**Chưa từng chạy thử trên Colab.** Code đã kiểm trên laptop (Python 3.13.5). Nếu Colab báo lỗi do khác phiên bản (joblib, pandas), gửi lỗi về để sửa.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/rsf'
import os
os.makedirs(f'{BASE}/results', exist_ok=True)
os.environ['RSF_PROCESSED_DIR'] = f'{BASE}/data'
os.environ['RSF_RESULTS_DIR'] = f'{BASE}/results'
print(os.listdir(f'{BASE}/data'))

Lấy code. Repo private thì dùng token GitHub (nhập bằng `getpass`, không dán vào cell), hoặc upload file zip của repo.

In [ ]:
import getpass, subprocess
REPO = 'github.com/myuyen0304/retail-sales-forecasting-research.git'
if not os.path.exists('/content/rsf'):
    token = getpass.getpass('GitHub token (để trống nếu repo public): ')
    url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
    subprocess.run(['git', 'clone', '--depth', '1', url, '/content/rsf'], check=True)
%cd /content/rsf
!git log -1 --oneline

Cài thư viện. Colab có sẵn numpy/pandas/torch, nên không ghim version ở đây; `run.py` ghi version thực tế vào `results/env_*.json`.

In [ ]:
!pip -q install -e . statsmodels lightgbm xgboost prophet
!python -m pytest -q

## 1. Tinh chỉnh (TOTAL / revenue, 4 fold)
RNN/LSTM tốn thời gian nhất: chạy riêng nếu phiên sắp hết.

In [ ]:
!python -m rsf.run tune --models sarima,sarimax,prophet,rf,gb,xgboost,lightgbm --jobs 2
!python -m rsf.run tune --models rnn,lstm --jobs 2
!cat $RSF_RESULTS_DIR/best_params.json

## 2. Cross-validation (4 chuỗi × 2 target × 4 fold)

In [ ]:
!python -m rsf.run cv --models baselines,main,ablation --jobs 2
!python -m rsf.run cv --models pure --jobs 2 --seeds 3

## 3. Bảng và hình CV
Dừng ở đây, gửi `results/tables` cho nhóm để chốt phương pháp đề xuất **trước khi** chạy test.

In [ ]:
!python scripts/make_tables.py && python scripts/make_figures.py
import pandas as pd
t = pd.read_csv(f"{os.environ['RSF_RESULTS_DIR']}/tables/T1_main.csv")
t[(t.series == 'TOTAL') & (t.target == 'revenue')][['model', 'mae_mean', 'mae_sd', 'rmse_mean', 'mape_mean', 'group']]

## 4. Test (chỉ chạy một lần, sau khi đã chốt)

In [ ]:
# !python -m rsf.run test --models all --jobs 2
# !python scripts/make_tables.py --test && python scripts/make_figures.py --test